# IT7075 MP1 — Colab Environment Verification

Covers three competencies in one session:

- **§3.4** Hosted notebooks and persistent storage — Drive mounted, file written, survives a runtime restart
- **§3.5** Programmatic model access — key from Colab Secrets, never from code
- **§3.8** Compute inventory — system utility *and* framework query, Colab side

**Before you start:** set the runtime to a GPU. `Runtime → Change runtime type → T4 GPU`.
A CPU-only runtime still earns full credit if reported accurately, but a GPU gives you a
contrast against your local RTX 5070 that's worth a line in the report.

**Screenshot discipline:** every cell below is designed so its output is safe to capture.
No credential value is ever printed.


---
## Part 1 — Compute inventory (§3.8, Colab half)

The rubric requires **both** a system utility and a framework query on each platform.


In [ ]:
# System utility
!nvidia-smi || echo 'nvidia-smi unavailable — this runtime has no GPU attached.'

In [ ]:
# Framework query + CPU/RAM, mirroring the local env_check.py output
import platform, os, sys

print(f'Platform : {platform.system()} {platform.release()}')
print(f'Python   : {platform.python_version()}')
print(f'Executable: {sys.executable}')
print(f'Cores    : {os.cpu_count()}')

try:
    import psutil
    print(f'Total RAM: {psutil.virtual_memory().total / (1024**3):.1f} GiB')
except ImportError:
    print('Total RAM: psutil unavailable')

try:
    import torch
    print(f'torch    : {torch.__version__}')
    print(f'torch.cuda.is_available() -> {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'  device 0: {torch.cuda.get_device_name(0)}')
        print(f'  CUDA build: {torch.version.cuda}')
    else:
        print('  No CUDA device — CPU-only runtime (this is a valid, full-credit result).')
except ImportError:
    print('torch not installed in this runtime')

---
## Part 2 — Google Drive persistence (§3.4), **step 1 of 2**

Mounts Drive, writes a timestamped file, and reads it straight back.
The timestamp is the point: after the restart it must still show the *original* write time,
which is what proves the file outlived the runtime rather than being recreated.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

folder = Path('/content/drive/MyDrive/IT7075')
folder.mkdir(parents=True, exist_ok=True)

target = folder / 'persistence_check.txt'
written_at = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
target.write_text(f'IT7075 MP1 persistence check\nWritten at: {written_at}\n', encoding='utf-8')

print(f'Wrote   : {target}')
print(f'Contents: {target.read_text(encoding="utf-8").strip()}')
print()
print('Session disk vs Drive:')
!ls -la /content/drive/MyDrive/IT7075/

---
# ⛔ STOP — restart the runtime now

`Runtime → Restart session` (**not** "Disconnect and delete runtime" — that would also wipe Drive's mount state
and make the next cells slower, though the file itself survives either way).

Wait for the runtime to come back, then continue with **step 2 below**. Do not re-run anything above.


## Part 2 — step 2 of 2, after the restart

Re-mount and read the file back. The `Written at:` line should show the time from *before* the restart,
and `ls -la` should show an mtime that predates the current session.

**This is the §3.4 screenshot.** Capture the mount confirmation and this output together.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone

target = Path('/content/drive/MyDrive/IT7075/persistence_check.txt')

print(f'Runtime restarted. Current time: {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'File still present: {target.exists()}')
print()
print('--- file contents, written before the restart ---')
print(target.read_text(encoding='utf-8').strip())
print('------------------------------------------------')
print()
!ls -la /content/drive/MyDrive/IT7075/

---
## Part 3 — Model access via Colab Secrets (§3.5)

**Set the secret first:** click the 🔑 key icon in the left sidebar → **Add new secret** →
name it exactly `OPENAI_API_KEY`, paste the value, and toggle **Notebook access** on.

The key is read from the Secrets store at runtime. It never appears in a cell, in the notebook file,
or in any output — which matters because cell output is saved *into* the .ipynb, and the .ipynb
gets committed.


In [ ]:
!pip install --quiet openai

In [ ]:
import os
from google.colab import userdata

# Load from Colab Secrets — never from a literal in the notebook
try:
    key = userdata.get('OPENAI_API_KEY')
except Exception as e:
    raise SystemExit(f'Could not read the secret. Is it named OPENAI_API_KEY with notebook access enabled? ({e})')

# Validate SHAPE, not just presence — a copied placeholder is non-empty and would
# otherwise sail past a simple `if not key` guard and fail later as a raw traceback.
if not key or key.startswith('sk-...') or key.endswith('...'):
    raise SystemExit('OPENAI_API_KEY is missing or is still a placeholder.')

print(f'Credential loaded from Colab Secrets: {len(key)} chars (value not shown)')

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=key)

try:
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user',
                   'content': 'Reply with one short sentence confirming API access works.'}],
        timeout=30,
    )
    print('Model response (Colab, key loaded from Secrets):')
    print(resp.choices[0].message.content)
except Exception as e:
    # No bare traceback: a 429 quota error or a blocked network should read as a diagnosis.
    print(f'API call failed: {type(e).__name__}: {e}')
    print('Common causes: no billing credit on the account, a spend limit already reached,')
    print('or the key was revoked. Check platform.openai.com → Usage.')

---
## Done

Screenshots to take from this notebook:

| Shot | Covers |
|---|---|
| Part 1 output (both cells) | §3.8 Colab half — system utility + framework query |
| Part 2 step 2 output | §3.4 — mount confirmation + file read back after restart |
| Part 3 output | §3.5 Colab half — model response, no key visible |

Then save a copy to Drive (`File → Save a copy in Drive`) so the executed notebook persists.
